# OULAD — Data Quality & Preparation

**Project:** OULA — Student Success Analytics  
**Phase:** 2 — Data quality and preparation

This notebook audits the seven raw OULAD tables before data are loaded into Django. The raw files are treated as the source of truth and are **not modified in place**.

### Main principles
- Missing values are investigated by meaning, not automatically filled.
- Exact duplicate rows are distinguished from repeated business keys with different measurements.
- Domain constraints are checked before any cleaning decision.
- Referential integrity is validated across the seven tables.
- Predictive-model preprocessing will be performed separately, using training data only, to avoid leakage.


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

# Expected local project structure:
# OULA-DjangoProject/data/raw/OULA.zip
ZIP_PATH = Path('../data/raw/OULA.zip')

if not ZIP_PATH.exists():
    candidates = [Path('../../data/raw/OULA.zip'), Path('OULA.zip')]
    for candidate in candidates:
        if candidate.exists():
            ZIP_PATH = candidate
            break

print('Dataset:', ZIP_PATH.resolve())
assert ZIP_PATH.exists(), 'Place OULA.zip in data/raw/ before running this notebook.'

zip_file = zipfile.ZipFile(ZIP_PATH)
RAW_PREFIX = 'anonymiseddata/'
FILES = {
    'assessments': RAW_PREFIX + 'assessments.csv',
    'courses': RAW_PREFIX + 'courses.csv',
    'studentAssessment': RAW_PREFIX + 'studentAssessment.csv',
    'studentInfo': RAW_PREFIX + 'studentInfo.csv',
    'studentRegistration': RAW_PREFIX + 'studentRegistration.csv',
    'studentVle': RAW_PREFIX + 'studentVle.csv',
    'vle': RAW_PREFIX + 'vle.csv',
}
print('Raw tables:', list(FILES))


## 1. Load the manageable tables

`studentVle.csv` has more than 10 million rows, so it is audited separately. The other six tables can be loaded normally.

In [ ]:
small_tables = {}
for name, member in FILES.items():
    if name != 'studentVle':
        small_tables[name] = pd.read_csv(zip_file.open(member))
        print(f'{name:20s}', small_tables[name].shape)


## 2. Basic structure and missingness

A missing value is not automatically an error. For example, an unregistration date can legitimately be missing. We record missingness first and decide treatment later, at the level of the analysis that needs the variable.

In [ ]:
quality_rows = []
for name, df in small_tables.items():
    for column in df.columns:
        quality_rows.append({
            'table': name,
            'column': column,
            'dtype': str(df[column].dtype),
            'missing_count': int(df[column].isna().sum()),
            'missing_pct': round(float(df[column].isna().mean() * 100), 3),
            'unique_count': int(df[column].nunique(dropna=True)),
        })
missing_report = pd.DataFrame(quality_rows)
missing_report[missing_report['missing_count'] > 0].sort_values(['table','missing_count'], ascending=[True,False])


### Current interpretation of important missing values

- `studentInfo.imd_band`: keep missing values initially. For a future ML model, test a categorical `Unknown` category versus train-only imputation and compare validation performance.
- `studentRegistration.date_unregistration`: missing is meaningful and should remain `NULL`/`NaN`.
- `assessments.date`: missing exam dates are preserved; we do not invent dates.
- `studentAssessment.score`: do **not** turn missing into 0 automatically. A missing score is different from an observed score of zero.
- `vle.week_from` / `week_to`: preserve missing values until an analysis specifically requires resource-week information.

For predictive modelling, preprocessing will be fitted on the training split only. Scikit-learn provides simple, KNN and iterative imputers and missingness indicators; the correct choice depends on feature semantics and validation results.

## 3. Duplicate checks on the six manageable tables

In [ ]:
duplicate_report = []
for name, df in small_tables.items():
    duplicate_report.append({
        'table': name,
        'rows': len(df),
        'exact_duplicate_rows_to_remove': int(df.duplicated().sum()),
        'rows_marked_as_part_of_duplicate_groups': int(df.duplicated(keep=False).sum()),
    })
pd.DataFrame(duplicate_report)


## 4. `studentVle.csv`: exact duplicates vs repeated interaction keys

`DataFrame.drop_duplicates()` compares **all columns by default**. Therefore, two rows with the same student, resource and day but different `sum_click` values are not exact duplicates. Using `drop_duplicates(subset=[...])` would impose a new business key and could discard valid click measurements.

The source documentation defines `sum_click` as the number of times a student interacted with a material on that day. Therefore, only truly identical rows are candidates for generic duplicate removal; repeated student-resource-day keys require an explicit aggregation decision.

In [ ]:
SVLE_COLS = ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']
KEY_COLS = ['code_module', 'code_presentation', 'id_student', 'id_site', 'date']

# 10.6M rows can require substantial RAM. Run this cell only on a machine
# with enough memory; otherwise use chunked processing for production cleaning.
student_vle = pd.read_csv(zip_file.open(FILES['studentVle']), usecols=SVLE_COLS)

svle_exact_duplicate_rows = int(student_vle.duplicated().sum())
svle_exact_duplicate_group_rows = int(student_vle.duplicated(keep=False).sum())
svle_unique_rows = int(student_vle.drop_duplicates().shape[0])
key_duplicate_group_count = int((student_vle.groupby(KEY_COLS).size() > 1).sum())
key_rows_to_remove_if_forced = int(student_vle.duplicated(subset=KEY_COLS).sum())
unique_business_keys = int(student_vle[KEY_COLS].drop_duplicates().shape[0])
click_counts_per_key = student_vle.groupby(KEY_COLS, sort=False)['sum_click'].nunique()
keys_with_multiple_click_values = int((click_counts_per_key > 1).sum())

print('Rows:', len(student_vle))
print('Exact duplicate rows removed by drop_duplicates():', svle_exact_duplicate_rows)
print('Rows belonging to exact-duplicate groups:', svle_exact_duplicate_group_rows)
print('Rows remaining after exact de-duplication:', svle_unique_rows)
print('Repeated student-resource-day keys:', key_duplicate_group_count)
print('Rows that subset-based de-duplication would discard:', key_rows_to_remove_if_forced)
print('Unique student-resource-day keys:', unique_business_keys)
print('Repeated keys with >1 distinct sum_click:', keys_with_multiple_click_values)


### Decision

For raw-to-processed cleaning, **only exact duplicate rows are candidates for removal**. We preserve repeated business-key rows with different `sum_click` values until we establish the appropriate aggregation grain for the Django `Interaction` model. If the application ultimately wants one row per student + course + resource + day, we can aggregate `sum_click` by that key and document that aggregation explicitly; that is a modelling transformation, not generic duplicate deletion.

## 5. Domain-range validation

We do not 'fix' negative clicks or invalid scores when none exist. A quality rule is still valuable when it passes: it becomes a test that future transformations can rerun.

In [ ]:
student_assessment = small_tables['studentAssessment']
invalid_scores = student_assessment['score'].notna() & ~student_assessment['score'].between(0, 100)
assessments = small_tables['assessments']
invalid_weights = assessments['weight'].notna() & ~assessments['weight'].between(0, 100)
negative_clicks = student_vle['sum_click'].lt(0)
range_report = pd.DataFrame({
    'check': [
        'studentAssessment.score outside [0,100]',
        'assessments.weight outside [0,100]',
        'studentVle.sum_click < 0',
    ],
    'invalid_count': [int(invalid_scores.sum()), int(invalid_weights.sum()), int(negative_clicks.sum())]
})
range_report


### Statistical and logical interpretation

- **Score:** 0 is a valid observation. Values below 0 or above 100 violate the documented measurement scale and should be investigated rather than clipped.
- **Clicks:** `sum_click` is a count of interactions, so a negative value has no meaningful interpretation under the variable definition. We would flag it and trace the source before correcting or removing it.
- **Current result:** no invalid values are present in the raw file, so there is nothing to repair here. This is a successful validation, not an unfinished cleaning task.

## 6. Missing assessment scores

The current OULAD data contain 173 missing `score` values, all associated with TMA records. We preserve them. For an achieved-score analysis, missing scores should not be silently treated as zero. For early warning, missingness may itself be informative, but only if it is known at the prediction time.

In [ ]:
assessment_meta = small_tables['assessments'][['id_assessment','assessment_type','code_module','code_presentation','weight','date']]
missing_score_detail = (student_assessment.loc[student_assessment['score'].isna()].merge(assessment_meta, on='id_assessment', how='left'))
missing_score_detail.groupby('assessment_type', dropna=False).size().rename('missing_scores')


## 7. Referential integrity

These checks verify that foreign-key-like identifiers used by one table exist in the corresponding parent table. This is a prerequisite for safe Django relationships.

In [ ]:
courses = small_tables['courses']
assessments = small_tables['assessments']
student_info = small_tables['studentInfo']
vle = small_tables['vle']

def missing_fk(child, child_cols, parent, parent_cols):
    parent_keys = parent[parent_cols].drop_duplicates()
    merged = child[child_cols].drop_duplicates().merge(parent_keys, left_on=child_cols, right_on=parent_cols, how='left', indicator=True)
    return int((merged['_merge'] == 'left_only').sum())

checks = {
    'studentAssessment.id_assessment -> assessments.id_assessment': missing_fk(student_assessment, ['id_assessment'], assessments, ['id_assessment']),
    'studentVle.id_site+course -> vle.id_site+course': missing_fk(student_vle, ['id_site','code_module','code_presentation'], vle, ['id_site','code_module','code_presentation']),
    'studentVle.student+course -> studentInfo.student+course': missing_fk(student_vle, ['id_student','code_module','code_presentation'], student_info, ['id_student','code_module','code_presentation']),
    'studentVle.course -> courses.course': missing_fk(student_vle, ['code_module','code_presentation'], courses, ['code_module','code_presentation']),
}
pd.Series(checks, name='orphan_distinct_keys')


## 8. Preparation decisions before Django

### Keep
- raw OULAD files unchanged
- meaningful missing values as `NaN` / `NULL`
- valid zero scores
- repeated interaction rows with different measurements

### Remove / transform later
- exact duplicate rows can be removed in a reproducible processed-data step
- repeated student-resource-day records should be **aggregated only if the chosen application grain requires it**
- ML-specific imputation belongs in the modelling pipeline, fitted on training data only

### Important modelling rule
`final_result` is an outcome. It must not be used to create an early-warning feature when the model is supposed to predict risk before the outcome is known. Likewise, post-outcome interactions or future assessments must not leak into earlier prediction windows.

## 9. Next phase

The data-quality audit gives us the evidence needed to design the Django models. Before importing 10.6 million interaction rows, we will decide the exact `Interaction` grain and write a reproducible preparation/import process with count checks. Then we can move to Django models and migrations.